# pandas Basics
## MAD 2502 — Applied Data

**Big idea:** A pandas DataFrame lets you inspect, filter, summarize, transform, and communicate tabular data.

This notebook is **standalone** and assumes only beginner Python.

By the end, you should be able to:

- explain what a DataFrame, row, column, and Series are;
- load a CSV;
- use `head()`, `shape`, `columns`, and `dtypes`;
- select columns and filter rows;
- identify missing values;
- compute basic summaries;
- make a grouped summary and a simple graph;
- save a result.

> **Lesson rhythm:** Understand → Predict → Run → Change → Use → Practice → Connect

[View on GitHub](https://github.com/mathguy-uf/MAD2502/blob/main/applied_data/02_pandas_basics.ipynb) · [Run in Google Colab](https://colab.research.google.com/github/mathguy-uf/MAD2502/blob/main/applied_data/02_pandas_basics.ipynb) · [Launch Binder/JupyterLab](https://mybinder.org/v2/gh/mathguy-uf/MAD2502/main?urlpath=lab/tree/applied_data/02_pandas_basics.ipynb)

Run cells from the top with **Shift+Enter**. If results become confusing, restart the kernel and run all cells from the top.

The loader below works from the repository in local Jupyter/Binder and falls back to the same CSV on GitHub when the local file is not present, as in a fresh Colab session.

In [ ]:
# If an import fails, uncomment and run this once.
# %pip install pandas matplotlib

## 1. Understand — What is a DataFrame?

We will use a small fictional campus-events dataset.

A pandas **DataFrame** is a table.

- A **row** is one record. Here, **one row represents one event**.
- A **column** stores one variable, such as `attendance`, `category`, or `cost`.
- A **Series** is one labeled column from a DataFrame.

Before analyzing a dataset, ask:

1. What does one row represent?
2. What does each column mean?
3. Which columns are numeric?
4. Are values missing?

A CSV is a text file that stores table data, usually with commas separating values.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

RAW_URL = (
    "https://raw.githubusercontent.com/"
    "mathguy-uf/MAD2502/main/"
    "applied_data/data/campus_events.csv"
)

path_from_repo_root = Path("applied_data/data/campus_events.csv")
path_from_this_folder = Path("data/campus_events.csv")

if path_from_repo_root.exists():
    data_source = path_from_repo_root
elif path_from_this_folder.exists():
    data_source = path_from_this_folder
else:
    data_source = RAW_URL

events = pd.read_csv(data_source)

print("data source:", data_source)
display(events.head())

### What just happened?

`pd.read_csv(...)` reads a CSV into a DataFrame.

The setup first checks two common local locations. If neither file exists, it uses the raw GitHub URL. This makes the notebook usable in:

- a local clone of the repository;
- Binder/JupyterLab;
- Google Colab;
- most Jupyter environments with internet access.

You do not need to memorize the setup code. The important idea is that `events` is now a DataFrame.

## 2. Predict — Inspect before analyzing

Without running the next cell, predict what each expression tells you:

```python
events.head(3)
events.shape
events.columns
events.dtypes
events["category"]
events[["event_name", "attendance"]]
```

Remember:

- `head(3)` shows the first three rows;
- `shape` is `(number_of_rows, number_of_columns)`;
- `columns` lists the column names;
- `dtypes` shows the stored type of each column;
- one column name in brackets returns a Series;
- a list of column names in double brackets returns a DataFrame.

In [ ]:
print("shape:", events.shape)
print("columns:", list(events.columns))
print("column types:")
print(events.dtypes)
print("category object type:", type(events["category"]))

display(events.head(3))
display(events[["event_name", "attendance"]].head(3))

## 3. Run — Filter rows

A **filter** keeps rows that satisfy a condition.

Start with a column:

```python
events["attendance"]
```

Compare each value:

```python
events["attendance"] >= 100
```

That produces `True` or `False` for every row. Put the condition inside `events[...]` to keep only the `True` rows.

A missing attendance value does not satisfy `>= 100`.

**Predict:** will the first filter keep many rows or only a few?

In [ ]:
large_events = events[events["attendance"] >= 100]
display(large_events[["event_name", "attendance"]])

free_outdoor = events[(events["cost"] == 0) & (events["location_type"] == "Outdoor")]

display(free_outdoor[["event_name", "cost", "location_type"]])

For pandas conditions:

- `&` means **AND**;
- `|` means **OR**;
- put each comparison in parentheses.

Do not use Python's words `and` and `or` for these element-by-element DataFrame filters.

## 4. Change — Make controlled changes

Change one thing at a time and predict the effect before rerunning.

1. Change `attendance >= 100` to `attendance >= 150`. Should the result get longer or shorter?
2. Change `&` to `|` in the free/outdoor filter. What changes?
3. Change one category name in the filter below.
4. Change `ascending=False` to `ascending=True`.

In [ ]:
music_or_sports = events[
    (events["category"] == "Music") | (events["category"] == "Sports")
]

display(music_or_sports)

highest_attendance = events.sort_values(
    "attendance",
    ascending=False,
)[["event_name", "attendance"]].head(5)

display(highest_attendance)

`sort_values(...)` changes the row order; it does not change the measurements themselves.

## 5. Use — Missing values and basic summaries

Before computing a result, inspect missing values.

`isna()` marks missing entries with `True`. Summing those `True` values counts how many are missing in each column.

A missing value means **unknown**, not automatically zero.

In [ ]:
missing_by_column = events.isna().sum()
display(missing_by_column)

### Basic numeric summaries

For a numeric column:

- `.count()` counts known values;
- `.mean()` computes the average of known values;
- `.median()` finds the middle known value;
- `.min()` and `.max()` find the smallest and largest known values;
- `.describe()` provides several common summaries at once.

Most pandas summary methods ignore missing numeric values by default, so always check how much data are missing.

In [ ]:
attendance = events["attendance"]

print("known attendance values:", attendance.count())
print("mean attendance:", attendance.mean())
print("median attendance:", attendance.median())
print("minimum attendance:", attendance.min())
print("maximum attendance:", attendance.max())

display(attendance.describe())

### Summaries for categories

`value_counts()` counts how often each distinct value appears.

This is useful for text columns such as `category`.

In [ ]:
category_counts = events["category"].value_counts(dropna=False)
display(category_counts)

print(
    "category count total:",
    category_counts.sum(),
    "rows:",
    len(events),
)

### Filling a missing value is a choice

For demonstration, the next cell creates a **new** column in which missing ratings are replaced by the median known rating.

This is called **imputation**. It is a modeling choice, not a universal rule. We keep the original `rating` column so the missingness is not hidden.

In [ ]:
rating_median = events["rating"].median()

events["rating_filled"] = events["rating"].fillna(rating_median)

print("missing original ratings:", events["rating"].isna().sum())
print("missing filled ratings:", events["rating_filled"].isna().sum())

### Answer a grouped question

Now ask:

> **Which event categories tend to have larger attendance?**

`groupby("category")` puts rows with the same category together.

Inside `.agg(...)`:

- `size` counts all rows in each category;
- `count` counts known attendance values;
- `mean` averages known attendance values;
- `median` finds a middle value.

Always read an average together with the number of observations supporting it.

In [ ]:
category_summary = (
    events.groupby(
        "category",
        as_index=False,
    )
    .agg(
        number_of_events=("event_name", "size"),
        known_attendance=("attendance", "count"),
        mean_attendance=("attendance", "mean"),
        median_cost=("cost", "median"),
    )
    .sort_values(
        "mean_attendance",
        ascending=False,
    )
)

display(category_summary)

### Communicate with a graph

The next cell makes a horizontal bar chart.

- `x="category"` chooses the labels.
- `y="mean_attendance"` chooses the bar lengths.
- `kind="barh"` means horizontal bar chart.

**Predict:** which category will have the longest bar? Then compare that category with its `known_attendance` value.

In [ ]:
plot_data = category_summary.sort_values("mean_attendance")

ax = plot_data.plot(
    x="category",
    y="mean_attendance",
    kind="barh",
    legend=False,
    figsize=(7, 4),
)

ax.set(
    xlabel="Mean attendance",
    ylabel="Event category",
    title="Mean attendance by event category",
)

plt.tight_layout()
plt.show()

These events are fictional, so the graph is practice rather than a finding about the University of Florida.

### Optional: save your result

A summarized table can become the input to the next stage of a project.

In [ ]:
output_path = Path("category_summary.csv")

category_summary.to_csv(
    output_path,
    index=False,
)

print("saved:", output_path.resolve())

## 6. Practice

Use `events` to answer each question.

1. What does one row represent?
2. How many rows and columns are in the DataFrame?
3. Which columns are numeric?
4. Which columns have missing values?
5. Which event had the highest attendance?
6. How many events were free?
7. What was the median attendance for outdoor events?
8. Which category had the highest average rating?
9. Create a table containing only events with attendance above the overall median.
10. Use `value_counts()` on one categorical column.
11. Make one labeled graph that answers a question about the dataset.

## 7. Connect

The basic pandas workflow is:

**inspect → ask → select → compute → check → communicate**

Before using a complicated method, make sure you can answer:

- What does one row represent?
- What do the columns mean?
- What values are missing?
- What rows did the filter keep or remove?
- What does the summary actually measure?
- Does the graph answer the question?

This pattern appears throughout the capstone.

**Next connection:** the Statistics Villains notebook adds one new challenge — getting a table from a webpage before doing the pandas analysis.